# RDD2022 India training and measurement

Run these cells on a Colab or Kaggle GPU runtime. The dataset download is about 502 MB, and full training is intentionally left to the hosted GPU.

In [ ]:
!git clone <REPOSITORY_URL> road-damage-detector
%cd road-damage-detector

In [ ]:
%pip install -r requirements.txt
%pip install ultralytics

## Prepare the dataset

The script downloads the India archive, converts mapped VOC annotations to YOLO labels, creates an 85/15 train/validation split, and selects the darkest validation images as `val_night`.

In [ ]:
!python prepare_dataset.py

In [ ]:
!python prepare_dataset.py --dry-run --limit 25

## Train

This produces the checkpoint used by the measurement cells.

In [ ]:
!python train.py --data rdd2022.yaml --epochs 100 --device 0

## Evaluate overall and night validation splits

The second command appends a separate headed run to the same Markdown report.

In [ ]:
CHECKPOINT = 'runs/road_damage/rdd2022_india/weights/best.pt'
!python eval.py --weights $CHECKPOINT --data rdd2022.yaml --output RESULTS.md
!python eval.py --weights $CHECKPOINT --data rdd2022_night.yaml --output RESULTS.md --append

In [ ]:
from pathlib import Path
sample_images = sorted(Path('data/images/val').glob('*'))
assert sample_images, 'No validation images were prepared'
sample_image = sample_images[0]
print(sample_image)
!python benchmark.py --weights $CHECKPOINT --image {sample_image} --device 0 --output BENCHMARK.md